# 아산시 방송 홍보 효과 분석
> 순수 방송효과 분리를 위한 인과추론 파이프라인

**구성:**
1. 설정 & 패키지
2. 카드매출 베이스라인 구축
3. T맵 관광지 방문 분석
4. 인과추론 (STL / DID / CausalImpact)
5. 시각화
6. 경제적 파급효과 산출

---
## 0. 패키지 설치 & 임포트

In [ ]:
# 필요시 설치
# !pip install pandas numpy matplotlib tqdm statsmodels
# !pip install causalimpact  # 선택

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager, rc
import zipfile, glob, os, re
from pathlib import Path
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트
try:
    font_path = "C:/Windows/Fonts/malgun.ttf"
    font_name = font_manager.FontProperties(fname=font_path).get_name()
    rc('font', family=font_name)
except:
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.0f}'.format)

print('Ready')

---
## 1. 설정

In [ ]:
# ============================
# 경로 설정 (여기만 수정하면 됨)
# ============================
DATA_DIR = Path(r"C:\Users\HP\Desktop\01.데이터")
CARD_DIR = DATA_DIR / "02. 카드매출 데이터"
KCB_DIR  = DATA_DIR / "03. 신용정보 데이터"
TMAP_DIR = DATA_DIR / "04. 내비게이션 데이터"
POP_DIR  = DATA_DIR / "01. 인구 데이터"

OUTPUT_DIR = Path(r"C:\Users\HP\Desktop\02.분석결과")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("데이터:", DATA_DIR)
print("결과:", OUTPUT_DIR)
print("하위 폴더:", [d for d in os.listdir(DATA_DIR) if os.path.isdir(DATA_DIR/d)] if DATA_DIR.exists() else "경로 없음")

In [ ]:
# ============================
# 방송 이벤트 정의
# ============================
BROADCASTS = [
    {
        "name": "전국노래자랑 아산시편", "broadcaster": "KBS1", "genre": "음악예능",
        "air_date": "2025-06-08", "air_end_date": "2025-06-08",
        "episodes": 1, "rating": 6.5, "target_demo": "중장년",
        "budget_1000won": 3518,
        "locations": ["신정호 잔디광장"],
        "dong_names": ["온양5동"],
        "confounders": ["이순신축제(4~5월)"],
    },
    {
        "name": "전현무계획2", "broadcaster": "MBN", "genre": "여행버라이어티",
        "air_date": "2025-11-07", "air_end_date": "2025-11-07",
        "episodes": 1, "rating": 1.5, "target_demo": "2030",
        "budget_1000won": 50000,
        "locations": ["아산시 일원(숨은맛집/명소)"],
        "dong_names": [],
        "confounders": [],
    },
    {
        "name": "굿모닝 대한민국", "broadcaster": "KBS2", "genre": "교양정보",
        "air_date": "2025-11-12", "air_end_date": "2025-11-16",
        "episodes": 1, "rating": 0.55, "target_demo": "전연령",
        "budget_1000won": 20000,
        "locations": ["온양온천역 광장","족욕체험장","온양온천 전통시장","곡교천 은행나무길","현충사","피나클랜드"],
        "dong_names": ["온양1동","염치읍","신창면"],
        "confounders": ["가을단풍시즌(10~11월)"],
    },
    {
        "name": "6시 내고향", "broadcaster": "KBS1", "genre": "교양정보",
        "air_date": "2025-11-13", "air_end_date": "2025-11-13",
        "episodes": 1, "rating": 5.5, "target_demo": "중장년",
        "budget_1000won": 110000,
        "locations": ["아산시 주요명소"],
        "dong_names": [],
        "confounders": [],
    },
    {
        "name": "박원숙의 같이삽시다 시즌3", "broadcaster": "KBS2", "genre": "예능",
        "air_date": "2025-11-24", "air_end_date": "2025-12-15",
        "episodes": 4, "rating": 3.0, "target_demo": "중장년",
        "budget_1000won": 133000,
        "locations": ["곡교천 은행나무길","신정호정원","영인산","외암민속마을","온양온천시장","도고파라다이스","아산스파포레"],
        "dong_names": ["염치읍","온양5동","영인면","송악면","온양1동","도고면","음봉면"],
        "confounders": ["가을단풍시즌","겨울온천시즌(12월)"],
    },
    {
        "name": "뛰어야산다2", "broadcaster": "MBN", "genre": "스포츠리얼버라이어티",
        "air_date": "2026-01-12", "air_end_date": "2026-01-12",
        "episodes": 1, "rating": 1.5, "target_demo": "2030",
        "budget_1000won": 45000,
        "locations": ["신정호정원","곡교천 은행나무길","현충사","온천","캠핑장"],
        "dong_names": ["온양5동","염치읍","온양1동"],
        "confounders": ["비수기(1월)"],
    },
    {
        "name": "황제성의 황제파워", "broadcaster": "SBS파워FM", "genre": "라디오공개방송",
        "air_date": "2026-05-09", "air_end_date": "2026-05-09",
        "episodes": 1, "rating": None, "target_demo": "전연령",
        "budget_1000won": 220000,
        "locations": ["온양온천역 주무대"],
        "dong_names": ["온양1동"],
        "confounders": ["이순신축제(4~5월)"],
    },
]

# 방송 요약
pd.DataFrame([{k: v for k, v in b.items() if k not in ['locations','dong_names','confounders']} for b in BROADCASTS])

In [ ]:
# ============================
# 코드 매핑 & 상수
# ============================
ASAN_DONG_MAP = {
    44200250: "염치읍", 44200253: "배방읍", 44200310: "송악면", 44200330: "탕정면",
    44200340: "신창면", 44200350: "음봉면", 44200360: "둔포면", 44200370: "영인면",
    44200380: "인주면", 44200390: "선장면", 44200400: "도고면",
    44200570: "온양1동", 44200580: "온양2동", 44200590: "온양3동",
    44200600: "온양4동", 44200610: "온양5동", 44200620: "온양6동",
}

# 관광 관련 업종 코드
CORE_TOURISM_BIZ = [1001, 1003, 1010, 1020, 2199, 7120]  # 숙박+레저
FOOD_BIZ = [8001, 8005, 8006, 8010, 8013, 8021, 8301]     # 음식점
RETAIL_BIZ = [4010, 4020, 4004]                             # 소매

# T맵 관광지 키워드 매핑
BROADCAST_POI_KEYWORDS = {
    "신정호": ["신정호", "신정호정원", "신정호공원"],
    "현충사": ["현충사"],
    "외암민속마을": ["외암", "외암민속"],
    "곡교천": ["곡교천", "은행나무길"],
    "온양온천": ["온양온천", "온천", "족욕", "온천시장"],
    "도고온천": ["도고", "도고파라다이스", "도고온천"],
    "피나클랜드": ["피나클랜드"],
    "영인산": ["영인산"],
    "아산스파포레": ["스파포레"],
    "세계꽃식물원": ["꽃식물원", "세계꽃"],
    "이순신관광체험센터": ["이순신", "이순신관광"],
}

WINDOW_PRE = 28   # 방송 전 4주
WINDOW_POST = 28  # 방송 후 4주

COLORS = ['#2196F3','#FF5722','#4CAF50','#FFC107','#9C27B0','#00BCD4','#E91E63']
print('설정 완료')

In [ ]:
# ============================
# 공통 함수
# ============================
def read_zip_csv(zip_path, csv_pattern=None, nrows=None, usecols=None):
    """ZIP 내부 CSV 읽기 (인코딩 자동 감지)"""
    with zipfile.ZipFile(zip_path) as zf:
        names = [n for n in zf.namelist() if n.endswith('.csv')]
        if csv_pattern:
            names = [n for n in names if csv_pattern in n]
        if not names:
            return None
        for enc in ['utf-8-sig','utf-8','cp949','euc-kr']:
            try:
                with zf.open(names[0]) as f:
                    return pd.read_csv(f, encoding=enc, nrows=nrows, usecols=usecols)
            except (UnicodeDecodeError, UnicodeError):
                continue
    return None

def extract_yyyymm(filepath):
    """파일명에서 YYYYMM 추출"""
    m = re.search(r'(\d{6})', os.path.basename(filepath))
    return m.group(1) if m else None

def match_poi(dstn_nm):
    """T맵 목적지명 → 관광지 매칭"""
    if pd.isna(dstn_nm): return None
    s = str(dstn_nm).lower()
    for poi, kws in BROADCAST_POI_KEYWORDS.items():
        for kw in kws:
            if kw in s: return poi
    return None

print('함수 정의 완료')

---
## 2. Step 1: 카드매출 베이스라인 구축

In [ ]:
# ZIP 파일 목록
card_zips = sorted(glob.glob(str(CARD_DIR / "**" / "*.zip"), recursive=True))
print(f"카드매출 ZIP: {len(card_zips)}개")
for z in card_zips[:5]:
    print(f"  {os.path.basename(z)} ({extract_yyyymm(z)})")
if len(card_zips) > 5:
    print(f"  ... 외 {len(card_zips)-5}개")

In [ ]:
# [디버그] 카드매출 ZIP 내부 파일 구조 확인
# 첫 번째 ZIP의 CSV 목록과 패턴 확인
import zipfile

if card_zips:
    print(f"첫 번째 ZIP: {os.path.basename(card_zips[0])}")
    with zipfile.ZipFile(card_zips[0]) as zf:
        for n in sorted(zf.namelist()):
            print(f"  {n} ({zf.getinfo(n).file_size/1024:.0f}KB)")
    
    # EXCL_RSDT 패턴 있는지 확인
    with zipfile.ZipFile(card_zips[0]) as zf:
        excl = [n for n in zf.namelist() if 'EXCL_RSDT' in n]
        cstmr = [n for n in zf.namelist() if 'CSTMR' in n]
        print(f"\nEXCL_RSDT 패턴: {excl}")
        print(f"CSTMR 패턴: {cstmr}")
    
    # 전체 ZIP 파일의 YYYYMM 목록
    all_ym = [extract_yyyymm(z) for z in card_zips]
    print(f"\n전체 기간: {sorted(set(all_ym))}")
    print(f"총 {len(card_zips)}개 ZIP")
else:
    print("카드매출 ZIP 없음 - 경로 확인: ", CARD_DIR)
    # 혹시 ZIP이 아니라 폴더별로 되어있나?
    print("\n하위 구조:")
    for root, dirs, files in os.walk(CARD_DIR):
        level = root.replace(str(CARD_DIR), '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        if level < 2:
            for f in files[:5]:
                print(f"{indent}  {f}")
            if len(files) > 5:
                print(f"{indent}  ... 외 {len(files)-5}개")

In [ ]:
# 2-1. 외지인 → 아산시 소비 데이터 추출
# EXCL_RSDT = 아산시 비거주자가 아산 내 가맹점에서 결제한 내역

monthly_outsider = []
weekly_outsider = []

for zpath in tqdm(card_zips, desc="외지인 소비 추출"):
    ym = extract_yyyymm(zpath)
    if not ym: continue

    # 월별 (읍면동)
    df = read_zip_csv(zpath, "EXCL_RSDT_CRTR_CCND_CSPT_DONG")
    if df is not None and len(df) > 0:
        df = df[df['FRCS_SGG_NM'] == '아산시'].copy()
        df['CRTR_YM'] = ym
        monthly_outsider.append(df)

    # 주별 (읍면동)
    df_wk = read_zip_csv(zpath, "WEEK_CCND_EXCL_RSDT_CRTR_CCND_CSPT_DONG")
    if df_wk is not None and len(df_wk) > 0:
        df_wk = df_wk[df_wk['FRCS_SGG_NM'] == '아산시'].copy()
        weekly_outsider.append(df_wk)

df_outsider_mm = pd.concat(monthly_outsider, ignore_index=True) if monthly_outsider else pd.DataFrame()
df_outsider_wk = pd.concat(weekly_outsider, ignore_index=True) if weekly_outsider else pd.DataFrame()

print(f"\n월별 외지인: {len(df_outsider_mm):,}행")
if len(df_outsider_mm) > 0:
    print(f"기간: {df_outsider_mm['CRTR_YM'].min()} ~ {df_outsider_mm['CRTR_YM'].max()}")
print(f"주별 외지인: {len(df_outsider_wk):,}행")

In [ ]:
# 2-2. 일별 외지인 매출 (ASAN_CSTMR_DATA에서)
daily_outsider = []

for zpath in tqdm(card_zips, desc="일별 CSTMR"):
    df = read_zip_csv(zpath, "ASAN_CSTMR_DATA")
    if df is None or len(df) == 0: continue
    # 아산시 거주자가 아닌 사람 = 외지인
    outsider = df[df['CTY_RGN_NM'] != '아산시'].copy()
    if len(outsider) == 0: continue
    agg = outsider.groupby(['SALE_DATE','ADMI_CTY_NM','TP_GRP_NM','TP_BUZ_NM']).agg(
        sale_amt=('SALE_AMT','sum'), sale_cnt=('SALE_CNT','sum')
    ).reset_index()
    daily_outsider.append(agg)

df_daily = pd.concat(daily_outsider, ignore_index=True) if daily_outsider else pd.DataFrame()
if len(df_daily) > 0:
    df_daily['SALE_DATE'] = pd.to_datetime(df_daily['SALE_DATE'].astype(str))
    print(f"일별 외지인: {len(df_daily):,}행, {df_daily['SALE_DATE'].min()} ~ {df_daily['SALE_DATE'].max()}")
    df_daily.to_csv(OUTPUT_DIR / "card_outsider_daily.csv", index=False, encoding='utf-8-sig')
else:
    print("일별 데이터 없음")

In [ ]:
# 2-3. 시계열 집계 & 저장
if len(df_outsider_mm) > 0:
    # 읍면동별 월별
    agg_dong = df_outsider_mm.groupby(['CRTR_YM','FRCS_DONG_NM']).agg(
        outsider_amt=('ALL_USE_AMT','sum'),
        outsider_cnt=('ALL_USE_NOCS','sum'),
        frcs_cnt=('FRCS_CNT','sum'),
    ).reset_index()
    agg_dong.to_csv(OUTPUT_DIR / "card_outsider_by_dong_monthly.csv", index=False, encoding='utf-8-sig')
    print(f"읍면동별 월별: {len(agg_dong):,}행 저장")
    display(agg_dong.head(10))

    # 업종별 월별
    agg_biz = df_outsider_mm.groupby(['CRTR_YM','TOBIZ_CD','TOBIZ_NM']).agg(
        outsider_amt=('ALL_USE_AMT','sum'),
        outsider_cnt=('ALL_USE_NOCS','sum'),
    ).reset_index()
    agg_biz.to_csv(OUTPUT_DIR / "card_outsider_by_biz_monthly.csv", index=False, encoding='utf-8-sig')
    print(f"업종별 월별: {len(agg_biz):,}행 저장")

    # 출발지별 월별
    agg_origin = df_outsider_mm.groupby(['CRTR_YM','CUST_CTPV_NM','CUST_SGG_NM']).agg(
        outsider_amt=('ALL_USE_AMT','sum'),
        outsider_cnt=('ALL_USE_NOCS','sum'),
    ).reset_index()
    agg_origin.to_csv(OUTPUT_DIR / "card_outsider_origin_monthly.csv", index=False, encoding='utf-8-sig')
    print(f"출발지별 월별: {len(agg_origin):,}행 저장")

In [ ]:
# 2-4. DID 패널 구축 (처치군 vs 대조군)
if len(df_outsider_mm) > 0:
    broadcast_dongs = set()
    for b in BROADCASTS:
        broadcast_dongs.update(b['dong_names'])
    print(f"방송 노출 읍면동 (처치군): {broadcast_dongs}")

    tourism_codes = set(CORE_TOURISM_BIZ + FOOD_BIZ)
    mask_tourism = df_outsider_mm['TOBIZ_CD'].isin(tourism_codes)
    mask_dong = df_outsider_mm['FRCS_DONG_NM'].isin(broadcast_dongs)

    # 처치군
    treat = df_outsider_mm[mask_tourism & mask_dong].groupby(['CRTR_YM','FRCS_DONG_NM','TOBIZ_NM']).agg(
        outsider_amt=('ALL_USE_AMT','sum'), outsider_cnt=('ALL_USE_NOCS','sum')
    ).reset_index()
    treat['group'] = 'treatment'

    # 대조군
    ctrl = df_outsider_mm[mask_tourism & ~mask_dong].groupby(['CRTR_YM','FRCS_DONG_NM','TOBIZ_NM']).agg(
        outsider_amt=('ALL_USE_AMT','sum'), outsider_cnt=('ALL_USE_NOCS','sum')
    ).reset_index()
    ctrl['group'] = 'control'

    df_did_panel = pd.concat([treat, ctrl], ignore_index=True)
    df_did_panel.to_csv(OUTPUT_DIR / "card_did_panel_monthly.csv", index=False, encoding='utf-8-sig')
    print(f"DID 패널: {len(df_did_panel):,}행 (처치: {len(treat):,} / 대조: {len(ctrl):,})")
    display(df_did_panel.groupby('group')['outsider_amt'].describe())

---
## 3. Step 2: T맵 관광지 방문 분석

In [ ]:
# T맵 로드
tmap_files = sorted(glob.glob(str(TMAP_DIR / "**" / "as_tmap_od_*.csv"), recursive=True))
print(f"T맵 파일: {len(tmap_files)}개")

chunks = []
for f in tqdm(tmap_files, desc="T맵 로드"):
    for enc in ['utf-8-sig','cp949']:
        try:
            chunks.append(pd.read_csv(f, encoding=enc))
            break
        except: continue

df_tmap = pd.concat(chunks, ignore_index=True)
df_tmap['drv_ymd'] = pd.to_datetime(df_tmap['drv_ymd'].astype(str))
print(f"전체: {len(df_tmap):,}행, {df_tmap['drv_ymd'].min()} ~ {df_tmap['drv_ymd'].max()}")

# 관광지 매칭
df_tmap['matched_poi'] = df_tmap['dstn_nm'].apply(match_poi)
df_tmap['is_tourism'] = df_tmap['dstn_ctgy'].str.startswith('여행/레저').fillna(False)
print(f"관광지 매칭: {df_tmap['matched_poi'].notna().sum():,}행")
print(f"관광카테고리: {df_tmap['is_tourism'].sum():,}행")
print(f"\n매칭 관광지 분포:")
display(df_tmap['matched_poi'].value_counts())

In [ ]:
# 관광지별 일별 시계열
poi_daily = df_tmap[df_tmap['matched_poi'].notna()].groupby(
    ['drv_ymd','matched_poi']
).agg(
    visit_cnt=('vst_cnt','sum'),
    avg_stay_min=('avg_stay_min','mean'),
    avg_drv_dstc=('avg_drv_dstc','mean'),
    unique_origins=('frst_dptre_sgg_nm','nunique'),
).reset_index()
poi_daily.to_csv(OUTPUT_DIR / "tmap_poi_daily.csv", index=False, encoding='utf-8-sig')
print(f"관광지별 일별: {len(poi_daily):,}행")
display(poi_daily.head())

# 전체 관광 일별
tourism_daily = df_tmap[df_tmap['is_tourism']].groupby('drv_ymd').agg(
    total_visits=('vst_cnt','sum'), total_pois=('dstn_nm','nunique'), avg_stay=('avg_stay_min','mean')
).reset_index()
tourism_daily.to_csv(OUTPUT_DIR / "tmap_tourism_daily.csv", index=False, encoding='utf-8-sig')
print(f"관광 전체 일별: {len(tourism_daily):,}행")

In [ ]:
# 출발지 분석
poi_origin = df_tmap[df_tmap['matched_poi'].notna()].groupby(
    ['drv_ymd','matched_poi','frst_dptre_ctpv_nm','frst_dptre_sgg_nm']
).agg(visit_cnt=('vst_cnt','sum')).reset_index()

poi_origin['ym'] = poi_origin['drv_ymd'].dt.to_period('M').astype(str)
origin_monthly = poi_origin.groupby(['ym','matched_poi','frst_dptre_ctpv_nm']
).agg(visit_cnt=('visit_cnt','sum')).reset_index().sort_values(
    ['ym','matched_poi','visit_cnt'], ascending=[True,True,False])

origin_monthly.to_csv(OUTPUT_DIR / "tmap_poi_origin_monthly.csv", index=False, encoding='utf-8-sig')
print(f"출발지별: {len(origin_monthly):,}행")
print("\nTop 출발 시도 (전체 기간):")
display(poi_origin.groupby('frst_dptre_ctpv_nm')['visit_cnt'].sum().sort_values(ascending=False).head(10))

In [ ]:
# 방송 전후 비교 요약
pre_post_results = []
for b in BROADCASTS:
    air = pd.Timestamp(b['air_date'])
    air_end = pd.Timestamp(b['air_end_date'])

    # 해당 방송 관광지 매칭
    pois = []
    for loc in b['locations']:
        for poi, kws in BROADCAST_POI_KEYWORDS.items():
            if any(kw in loc for kw in kws) or poi in loc:
                pois.append(poi)
    mask_poi = df_tmap['matched_poi'].isin(pois) if pois else df_tmap['is_tourism']
    poi_label = "+".join(set(pois))[:30] if pois else "전체관광"

    pre_start = air - pd.Timedelta(days=WINDOW_PRE)
    pre = df_tmap[mask_poi & (df_tmap['drv_ymd'] >= pre_start) & (df_tmap['drv_ymd'] < air)]['vst_cnt'].sum()
    post = df_tmap[mask_poi & (df_tmap['drv_ymd'] > air_end) & (df_tmap['drv_ymd'] <= air_end + pd.Timedelta(days=WINDOW_POST))]['vst_cnt'].sum()

    pre_avg = pre / max(WINDOW_PRE, 1)
    post_avg = post / max(WINDOW_POST, 1)
    chg = (post_avg - pre_avg) / pre_avg * 100 if pre_avg > 0 else np.nan

    # 전년 동기
    yoy_air = air - pd.Timedelta(days=365)
    yoy = df_tmap[mask_poi & (df_tmap['drv_ymd'] >= yoy_air) & (df_tmap['drv_ymd'] <= yoy_air + pd.Timedelta(days=WINDOW_POST))]['vst_cnt'].sum()
    yoy_avg = yoy / max(WINDOW_POST, 1)
    yoy_chg = (post_avg - yoy_avg) / yoy_avg * 100 if yoy_avg > 0 else np.nan

    pre_post_results.append({
        'broadcast': b['name'], 'air_date': b['air_date'], 'poi': poi_label,
        'pre_daily_avg': round(pre_avg,1), 'post_daily_avg': round(post_avg,1),
        'change_pct': round(chg,1) if not np.isnan(chg) else None,
        'yoy_daily_avg': round(yoy_avg,1),
        'yoy_change_pct': round(yoy_chg,1) if not np.isnan(yoy_chg) else None,
    })

df_prepost = pd.DataFrame(pre_post_results)
df_prepost.to_csv(OUTPUT_DIR / "tmap_broadcast_effect_summary.csv", index=False, encoding='utf-8-sig')
display(df_prepost)

---
## 4. Step 3: 인과추론

In [ ]:
# 4-1. STL 계절 분해
from statsmodels.tsa.seasonal import STL

if len(df_outsider_mm) > 0:
    ts = df_outsider_mm.groupby('CRTR_YM')['ALL_USE_AMT'].sum().reset_index()
    ts.columns = ['ym', 'amt']
    ts['ym'] = pd.to_datetime(ts['ym'], format='%Y%m')
    ts = ts.set_index('ym').sort_index().asfreq('MS')

    if len(ts) >= 24:
        result = STL(ts['amt'], period=12, robust=True).fit()

        fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
        for ax, data, title in zip(axes,
            [result.observed, result.trend, result.seasonal, result.resid],
            ['관측값 (외지인 매출)', '추세', '계절성', '잔차 (이상치)']):
            ax.plot(data, color='#2196F3', linewidth=1.5)
            ax.set_title(title, fontweight='bold')
            ax.grid(True, alpha=0.3)
            # 방송 이벤트 표시
            for b in BROADCASTS:
                air = pd.Timestamp(b['air_date'])
                if ts.index.min() <= air <= ts.index.max():
                    ax.axvline(x=air, color='#FF5722', linestyle=':', alpha=0.4)

        # 잔차 2sigma 이상치
        mu, sigma = result.resid.mean(), result.resid.std()
        axes[3].axhline(y=mu+2*sigma, color='red', linestyle='--', alpha=0.5)
        axes[3].axhline(y=mu-2*sigma, color='red', linestyle='--', alpha=0.5)
        anomalies = result.resid[abs(result.resid - mu) > 2*sigma]
        axes[3].scatter(anomalies.index, anomalies, color='red', s=50, zorder=5)

        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "fig_stl_decomposition.png", dpi=150, bbox_inches='tight')
        plt.show()
        print(f"이상치 월: {list(anomalies.index.strftime('%Y-%m'))}")
    else:
        print(f"데이터 부족: {len(ts)}개월 (최소 24개월 필요)")

In [ ]:
# 4-2. DID (이중차분법)
try:
    df_panel = pd.read_csv(OUTPUT_DIR / "card_did_panel_monthly.csv", encoding='utf-8-sig')
except:
    df_panel = df_did_panel if 'df_did_panel' in dir() else pd.DataFrame()

if len(df_panel) > 0:
    did_results = []
    for b in BROADCASTS:
        air_ym = int(pd.Timestamp(b['air_date']).strftime('%Y%m'))
        panel = df_panel.copy()
        panel['post'] = (panel['CRTR_YM'].astype(int) >= air_ym).astype(int)
        panel['treat'] = (panel['group'] == 'treatment').astype(int)
        panel['did'] = panel['post'] * panel['treat']

        means = panel.groupby(['post','treat'])['outsider_amt'].mean().unstack()

        try:
            from statsmodels.formula.api import ols
            model = ols('outsider_amt ~ post * treat', data=panel).fit()
            coef = model.params.get('post:treat', np.nan)
            pval = model.pvalues.get('post:treat', np.nan)
        except:
            # 수동 계산
            try:
                coef = (means.loc[1,1]-means.loc[0,1]) - (means.loc[1,0]-means.loc[0,0])
                pval = np.nan
            except:
                coef, pval = np.nan, np.nan

        did_results.append({'broadcast': b['name'], 'did_effect': coef, 'p_value': pval})
        print(f"[{b['name']}] DID: {coef:,.0f}" + (f" (p={pval:.4f})" if not np.isnan(pval) else ""))

    df_did_res = pd.DataFrame(did_results)
    df_did_res.to_csv(OUTPUT_DIR / "did_results.csv", index=False, encoding='utf-8-sig')
    display(df_did_res)
else:
    print("DID 패널 없음")

In [ ]:
# 4-3. CausalImpact (선택 - 패키지 있을 때만)
try:
    from causalimpact import CausalImpact
    HAS_CI = True
    print("CausalImpact 사용 가능")
except ImportError:
    HAS_CI = False
    print("CausalImpact 미설치 - 건너뜀 (pip install causalimpact)")

if HAS_CI and len(poi_daily) > 0:
    ci_results = []
    for b in BROADCASTS:
        pois = []
        for loc in b['locations']:
            for poi, kws in BROADCAST_POI_KEYWORDS.items():
                if any(kw in loc for kw in kws) or poi in loc:
                    pois.append(poi)
        for poi in set(pois):
            ts = poi_daily[poi_daily['matched_poi']==poi].set_index('drv_ymd')['visit_cnt'].resample('D').sum().fillna(0)
            if len(ts) < 60: continue
            air = pd.Timestamp(b['air_date'])
            pre = [ts.index.min(), air - pd.Timedelta(days=1)]
            post = [air, min(air + pd.Timedelta(days=WINDOW_POST), ts.index.max())]
            if pre[1] <= pre[0] or post[1] <= post[0]: continue
            try:
                ci = CausalImpact(pd.DataFrame({'y': ts}), pre, post)
                print(f"[{b['name']}] {poi}:")
                print(ci.summary())
                ci_results.append({'broadcast': b['name'], 'poi': poi})
            except Exception as e:
                print(f"  {poi} 실패: {e}")

---
## 5. Step 4: 시각화

In [ ]:
# 5-1. 외지인 매출 타임라인 + 방송 이벤트
if len(df_outsider_mm) > 0:
    ts = df_outsider_mm.groupby('CRTR_YM')['ALL_USE_AMT'].sum().reset_index()
    ts.columns = ['date', 'amt']
    ts['date'] = pd.to_datetime(ts['date'].astype(str), format='%Y%m')
    ts = ts.sort_values('date')
    # 최근 2년만 (2024~)
    ts_viz = ts[ts['date'] >= '2024-01-01'].copy()
    if len(ts_viz) == 0:
        ts_viz = ts.tail(24)  # 데이터가 2024 이전뿐이면 최근 24개월

    fig, ax = plt.subplots(figsize=(16, 6))
    ax.bar(ts_viz['date'], ts_viz['amt']/1e9, width=20, color='#2196F3', alpha=0.7)
    ax.plot(ts_viz['date'], ts_viz['amt']/1e9, color='#FF5722', linewidth=2, marker='o', markersize=4)

    ymax = ts_viz['amt'].max()/1e9 * 1.1
    for i, b in enumerate(BROADCASTS):
        air = pd.Timestamp(b['air_date'])
        if len(ts_viz) > 0 and ts_viz['date'].min() <= air <= ts_viz['date'].max() + pd.Timedelta(days=60):
            c = COLORS[i%len(COLORS)]
            ax.axvline(x=air, color=c, linestyle='--', alpha=0.7, linewidth=1.5)
            ax.text(air, ymax*(0.95-i*0.07), f" {b['name'][:10]}\n ({b['air_date']})",
                    fontsize=7, color=c, va='top')

    ax.set_title('아산시 외지인 카드매출 추이 + 방송 이벤트', fontsize=14, fontweight='bold')
    ax.set_ylabel('매출액 (십억원)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    plt.xticks(rotation=45)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_card_timeline.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'데이터 기간: {ts["date"].min()} ~ {ts["date"].max()} ({len(ts)}개월)')
    print(f'차트 표시: {ts_viz["date"].min()} ~ {ts_viz["date"].max()} ({len(ts_viz)}개월)')
else:
    print('외지인 카드매출 데이터 없음')


In [ ]:
# 5-2. T맵 관광 방문 타임라인
if len(tourism_daily) > 0:
    # 2024년부터만 표시 (방송 전 1년 baseline + 방송 시점)
    tourism_daily = tourism_daily[tourism_daily['drv_ymd'] >= '2024-01-01'].copy()
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.plot(tourism_daily['drv_ymd'], tourism_daily['total_visits'], alpha=0.3, color='#2196F3')
    ma = tourism_daily['total_visits'].rolling(7, min_periods=1).mean()
    ax.plot(tourism_daily['drv_ymd'], ma, color='#FF5722', linewidth=2, label='7일 이동평균')

    ymax = tourism_daily['total_visits'].max()
    for i, b in enumerate(BROADCASTS):
        air = pd.Timestamp(b['air_date'])
        if tourism_daily['drv_ymd'].min() <= air <= tourism_daily['drv_ymd'].max():
            c = COLORS[i%len(COLORS)]
            ax.axvline(x=air, color=c, linestyle='--', alpha=0.7)
            ax.text(air, ymax*(0.95-i*0.08), f" {b['name'][:8]}", fontsize=7, color=c, va='top')

    ax.set_title('아산시 T맵 관광지 방문 추이 + 방송 이벤트', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_tmap_timeline.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 5-3. 방송 전후 비교 바 차트
if len(df_prepost) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    x = range(len(df_prepost))
    w = 0.35
    axes[0].bar([i-w/2 for i in x], df_prepost['pre_daily_avg'], w, label='방송 전 4주', color='#90CAF9')
    axes[0].bar([i+w/2 for i in x], df_prepost['post_daily_avg'], w, label='방송 후 4주', color='#FF8A65')
    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels(df_prepost['broadcast'], rotation=45, ha='right', fontsize=8)
    axes[0].set_title('방송 전후 일평균 관광지 방문 (T맵)', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')

    vals = df_prepost['change_pct'].fillna(0)
    colors = ['#4CAF50' if v > 0 else '#F44336' for v in vals]
    axes[1].barh(df_prepost['broadcast'], vals, color=colors)
    axes[1].axvline(x=0, color='black', linewidth=0.5)
    axes[1].set_title('방송 전후 변화율 (%)', fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='x')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_pre_post_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 5-4. 관광지별 시계열 (개별 POI)
if len(poi_daily) > 0:
    poi_daily_viz = poi_daily[poi_daily['drv_ymd'] >= '2024-01-01'].copy()
    pois = poi_daily_viz['matched_poi'].value_counts().head(6).index.tolist()
    fig, axes = plt.subplots(len(pois), 1, figsize=(16, 3*len(pois)), sharex=True)
    if len(pois) == 1: axes = [axes]

    for idx, poi in enumerate(pois):
        ax = axes[idx]
        ts = poi_daily_viz[poi_daily_viz['matched_poi']==poi].sort_values('drv_ymd')
        ax.plot(ts['drv_ymd'], ts['visit_cnt'], alpha=0.4, color='#2196F3', linewidth=0.8)
        if len(ts) > 7:
            ax.plot(ts['drv_ymd'], ts['visit_cnt'].rolling(7,min_periods=1).mean(),
                    color='#FF5722', linewidth=2)
        # 방송 표시
        for b in BROADCASTS:
            for loc in b['locations']:
                if poi.lower() in loc.lower() or any(kw in loc for kw in BROADCAST_POI_KEYWORDS.get(poi,[])):
                    ax.axvline(x=pd.Timestamp(b['air_date']), color='red', linestyle='--', alpha=0.8)
                    ax.text(pd.Timestamp(b['air_date']), ax.get_ylim()[1]*0.9,
                            b['name'][:6], fontsize=7, color='red', rotation=90, va='top')
        ax.set_title(poi, fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_poi_timeseries.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 5-5. 출발지 히트맵
if len(origin_monthly) > 0:
    pivot = origin_monthly.pivot_table(values='visit_cnt', index='frst_dptre_ctpv_nm',
                                       columns='ym', aggfunc='sum', fill_value=0)
    top = pivot.sum(axis=1).nlargest(15).index
    pivot = pivot.loc[top]

    fig, ax = plt.subplots(figsize=(16, 8))
    im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=90, fontsize=7)
    ax.set_title('출발지(시도)별 아산 관광지 방문 추이', fontsize=14, fontweight='bold')
    plt.colorbar(im, ax=ax, label='방문 수')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_origin_heatmap.png", dpi=150, bbox_inches='tight')
    plt.show()

---
## 6. Step 5: 경제적 파급효과 산출

In [ ]:
# 1인당 관광소비액 추정
try:
    agg_biz = pd.read_csv(OUTPUT_DIR / "card_outsider_by_biz_monthly.csv", encoding='utf-8-sig')
except:
    agg_biz = pd.DataFrame()

if len(agg_biz) > 0:
    avg_tx = agg_biz.groupby('TOBIZ_NM').agg(
        total_amt=('outsider_amt','sum'), total_cnt=('outsider_cnt','sum')
    ).reset_index()
    avg_tx['avg_per_tx'] = (avg_tx['total_amt']/avg_tx['total_cnt']).round(0)
    avg_tx = avg_tx.sort_values('total_amt', ascending=False)
    print("업종별 건당 평균 결제액:")
    display(avg_tx.head(15))

    overall_avg = (agg_biz['outsider_amt'].sum() / agg_biz['outsider_cnt'].sum())
    print(f"\n전체 건당 평균: {overall_avg:,.0f}원")
    TX_PER_PERSON = 3.5
    per_capita = overall_avg * TX_PER_PERSON
    print(f"1인당 일 소비 추정: {per_capita:,.0f}원 (= {overall_avg:,.0f} x {TX_PER_PERSON}건)")
else:
    per_capita = 80000
    print(f"데이터 없음 → fallback: {per_capita:,}원")

In [ ]:
# 프로그램별 경제효과
MULTIPLIER = 1.8  # 관광업 산업연관 승수

try:
    df_did_res = pd.read_csv(OUTPUT_DIR / "did_results.csv", encoding='utf-8-sig')
except:
    df_did_res = pd.DataFrame()

econ = []
for b in BROADCASTS:
    budget = b['budget_1000won'] * 1000

    # T맵 전후 비교에서
    row = df_prepost[df_prepost['broadcast']==b['name']]
    if len(row) > 0:
        pre_avg = row.iloc[0]['pre_daily_avg']
        post_avg = row.iloc[0]['post_daily_avg']
        chg_pct = row.iloc[0]['change_pct']
        daily_inc = max(post_avg - pre_avg, 0)
        total_inc = daily_inc * WINDOW_POST
    else:
        pre_avg = post_avg = chg_pct = daily_inc = total_inc = np.nan

    direct = total_inc * per_capita if not np.isnan(total_inc) else np.nan
    total_eff = direct * MULTIPLIER if not np.isnan(direct) else np.nan
    roi = (total_eff - budget) / budget * 100 if not np.isnan(total_eff) and budget > 0 else np.nan

    # DID
    did_val = np.nan
    if len(df_did_res) > 0:
        r = df_did_res[df_did_res['broadcast']==b['name']]
        if len(r) > 0: did_val = r.iloc[0]['did_effect']

    econ.append({
        '방송': b['name'],
        '방영일': b['air_date'],
        '장르': b['genre'],
        '시청률': b['rating'],
        '예산(원)': budget,
        '일평균방문증가': round(daily_inc,1) if not np.isnan(daily_inc) else None,
        '28일총방문증가': round(total_inc,0) if not np.isnan(total_inc) else None,
        '직접효과(원)': round(direct,0) if not np.isnan(direct) else None,
        '총효과(원)': round(total_eff,0) if not np.isnan(total_eff) else None,
        'ROI(%)': round(roi,1) if not np.isnan(roi) else None,
        'DID효과': round(did_val,0) if not np.isnan(did_val) else None,
        '전후변화(%)': chg_pct,
    })

df_econ = pd.DataFrame(econ)
df_econ.to_csv(OUTPUT_DIR / "economic_impact_by_broadcast.csv", index=False, encoding='utf-8-sig')
display(df_econ)

In [ ]:
# 업종별 파급 구조 (2025 상/하반기 비교)
if len(agg_biz) > 0:
    agg_biz['year'] = agg_biz['CRTR_YM'].astype(str).str[:4].astype(int)
    agg_biz['half'] = np.where(agg_biz['CRTR_YM'].astype(str).str[4:6].astype(int) <= 6, 'H1', 'H2')
    df25 = agg_biz[agg_biz['year']==2025]
    if len(df25) > 0:
        hc = df25.groupby(['TOBIZ_NM','half'])['outsider_amt'].sum().unstack(fill_value=0)
        if 'H1' in hc.columns and 'H2' in hc.columns:
            hc['변화율(%)'] = ((hc['H2']-hc['H1'])/hc['H1']*100).round(1)
            hc = hc.sort_values('H2', ascending=False).head(20)
            print("2025년 상/하반기 업종별 외지인 매출 비교 (상위 20):")
            display(hc)
    else:
        print("2025년 데이터 없음")

---
## 7. 종합 요약 & 결과 파일 목록

In [ ]:
print("=" * 60)
print("생성된 파일 목록")
print("=" * 60)
for f in sorted(OUTPUT_DIR.glob("*")):
    sz = f.stat().st_size / 1024
    print(f"  {f.name:45s} {sz:>8,.0f} KB")

print(f"\n총 {len(list(OUTPUT_DIR.glob('*')))}개 파일")
print(f"\n※ 보완 필요 사항:")
print(f"  - SKT 유동인구 2025년 데이터 → 방문객 직접 측정")
print(f"  - 아산페이 데이터 → 지역 내 소비 효과 정밀 측정")
print(f"  - 네이버/구글 검색 트렌드 → 온라인 관심도 교차검증")
print(f"  - 비교도시(천안/서산) 데이터 → 전국 트렌드 통제")